## Data Cleaning Principles

- Raw data is not modified; all transformations are applied on a copy.
- Returns and cancellations are **not deleted**, but flagged.
- Missing data is handled deliberately, not blindly removed.
- All assumptions are documented and justified.
- Business meaning is prioritized over aggressive data reduction.

### PHASE 1: Data Preparation

- Load year-wise CSV files

- Combine into a single dataset

- Create a working copy

- Record baseline row count

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df_2009 = pd.read_csv("/Users/hepigediya/Desktop/retail-omnichannel-analytics/data/raw/online_retail_2009_2010.csv")
df_2010 = pd.read_csv("/Users/hepigediya/Desktop/retail-omnichannel-analytics/data/raw/online_retail_2010_2011.csv")

In [3]:
df_raw = pd.concat([df_2009, df_2010], ignore_index=True)

In [4]:
df = df_raw.copy()

In [5]:
df.shape

(1067371, 8)

### PHASE 2: Schema Standardization

- Standardize column names (snake_case)

- Ensure correct data types

- Convert dates to datetime format

In [6]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

Standardize Column Names
 
 Rules we apply
- Lowercase
- Replace spaces with _
- Remove special characters
- Use snake_case

In [7]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)

Rename Key Columns (Explicit Mapping)

Some columns should be business-clear.

In [8]:
df = df.rename(columns={
    "invoice": "invoice_no",
    "stockcode": "stock_code",
    "invoicedate": "invoice_date",
    "price": "unit_price",
    "customerid": "customer_id"
})

In [9]:
df.columns

Index(['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date',
       'unit_price', 'customer_id', 'country'],
      dtype='object')

In [10]:
df.dtypes

invoice_no       object
stock_code       object
description      object
quantity          int64
invoice_date     object
unit_price      float64
customer_id     float64
country          object
dtype: object

In [14]:
df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['customer_id'] = pd.to_numeric(df['customer_id'], errors='coerce')

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice_no    1067371 non-null  object        
 1   stock_code    1067371 non-null  object        
 2   description   1062989 non-null  object        
 3   quantity      1067371 non-null  int64         
 4   invoice_date  1067371 non-null  datetime64[ns]
 5   unit_price    1067371 non-null  float64       
 6   customer_id   824364 non-null   Int64         
 7   country       1067371 non-null  object        
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 66.2+ MB


### PHASE 3: Returns & Cancellations Handling

- Identify negative quantity values

- Identify invoices starting with 'C'

- Create flags:

- is_return

- is_cancellation

- Do not remove these records

In [16]:
# Flag returned items based on negative quantity
df['is_return'] = df['quantity'] < 0

In [17]:
# Flag cancellation / credit invoices (invoice numbers starting with 'C')
df['is_cancellation'] = df['invoice_no'].astype(str).str.startswith('C')

In [22]:
# Count return and cancellation flags
df['is_return'].value_counts()

is_return
False    1044421
True       22950
Name: count, dtype: int64

In [23]:
df['is_cancellation'].value_counts()

is_cancellation
False    1047877
True       19494
Name: count, dtype: int64

In [24]:
# Count return and cancellation flags
df[['is_return', 'is_cancellation']].value_counts()

is_return  is_cancellation
False      False              1044420
True       True                 19493
           False                 3457
False      True                     1
Name: count, dtype: int64

In [25]:
# Understand relationship between returns and cancellations
pd.crosstab(df['is_return'], df['is_cancellation'])

is_cancellation,False,True
is_return,,
False,1044420,1
True,3457,19493


In [26]:
# Number of return rows
(df['quantity'] < 0).sum()

22950

In [ ]:
# Number of cancellation invoices
df['invoice_no'].astype(str).str.startswith('C').sum()

0          489434
1          489434
2          489434
3          489434
4          489434
            ...  
1067366    581587
1067367    581587
1067368    581587
1067369    581587
1067370    581587
Name: invoice_no, Length: 1067371, dtype: object

### PHASE 4: Customer Data Handling

- Identify missing customer_id

- Retain anonymous transactions

- Document impact on customer-level analysis

In [29]:
# Count missing customer IDs
df['customer_id'].isna().sum()

243007

In [30]:
# Flag anonymous customer transactions
df['is_anonymous_customer'] = df['customer_id'].isna()

In [31]:
# Distribution of anonymous vs known customers
df['is_anonymous_customer'].value_counts()

is_anonymous_customer
False    824364
True     243007
Name: count, dtype: int64

In [32]:
# Percentage of anonymous transactions
(df['is_anonymous_customer'].mean() * 100).round(2)

22.77

### PHASE 5: Pricing Validation

- Identify zero or negative unit_price

- Remove invalid pricing rows

- Justify removal as data quality issue

In [33]:
# Identify rows with zero or negative unit price
invalid_price_mask = df['unit_price'] <= 0

invalid_price_mask.sum()

6207

In [34]:
# Preview invalid pricing records
df.loc[invalid_price_mask].head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_return,is_cancellation,is_anonymous_customer
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,True,False,True
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,True,False,True
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,<NA>,United Kingdom,True,False,True
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom,True,False,True
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom,True,False,True


In [35]:
# Remove invalid pricing rows
df = df.loc[~invalid_price_mask].copy()

In [36]:
df.shape

(1061164, 11)

In [37]:
# Confirm all remaining prices are positive
(df['unit_price'] <= 0).sum()

0

### PHASE 6: Feature Engineering

- Create derived fields:

- sales_amount = quantity × unit_price

- order_date

- order_month

- order_year

### PHASE 7: Duplicate Validation

- Check for exact duplicate rows

- Remove duplicates only if confirmed

### PHASE 8: Channel Preparation (Later Use)

- Keep structure ready for sales_channel

- Channel logic will be derived in analysis phase

### PHASE 9: Validation & Export

- Validate row counts after each step

- Sanity-check key metrics

- Export cleaned dataset to:

data/processed/cleaned_retail_sales.csv